In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import polars as pl

from ethos.constants import PROJECT_ROOT
from ethos.constants import SpecialToken as ST
from ethos.inference.constants import Reason, Task
from ethos.metrics import compute_and_print_metrics, preprocess_inference_results

results_dir = PROJECT_ROOT / "results"

## ICU Admission results

In [ ]:
def process_icu_admission_results(input_dir, **kwargs) -> pl.DataFrame:
    return preprocess_inference_results(
        input_dir,
        actual_expr=pl.col("actual").is_in([ST.DEATH, ST.ICU_ADMISSION]),
        expected_expr=pl.col("expected").is_in([ST.DEATH, ST.ICU_ADMISSION]),
        filter_ambiguous=pl.col("stop_reason") == Reason.GOT_TOKEN,
        **kwargs,
    )

In [2]:
# DEBUGGING ONLY
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / 'pyproject.toml').exists() and project_root != project_root.parent:
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
from src.ethos.datasets.mimic_icu import ICUAdmissionDataset
path = "../../../../mnt/data_share/project_henri/ethos-ares/mimic-tokenized/test"
dataset = ICUAdmissionDataset(path)

21743 21743


In [16]:
sum_dc  = sum([True if d[1]['expected'] == 'HOSPITAL_DISCHARGE' else False for d in dataset])
sum_dth = sum([True if d[1]['expected'] == 'MEDS_DEATH' else False for d in dataset])
sum_adm = sum([True if d[1]['expected'] == 'ICU_ADMISSION' else False for d in dataset])

In [17]:
sum_adm/len(dataset)

0.14915145104171457

In [23]:
import json
true_ids = [d[1]['patient_id'] for d in dataset if d[1]['expected'] == 'ICU_ADMISSION']
with open("true_ids.json", "w") as f:
    json.dump(true_ids, f)


In [18]:
sum_adm

3243

In [ ]:
icu_admission_results = []
for input_dir in (results_dir / Task.ICU_ADMISSION).iterdir():
    df = process_icu_admission_results(input_dir)
    rep_num = df["counts"].mean()
    res = compute_and_print_metrics(
        df["expected"],
        df["actual"],
        f"ICU Admission (rep_num={rep_num:.2f})\n{input_dir.name}",
    )
    icu_admission_results.append(
        {
            "name": input_dir.name,
            "auc": res["fitted_auc"],
            "rep_num": rep_num,
        }
    )
    plt.show()
icu_admission_results = pl.DataFrame(icu_admission_results).sort("auc", descending=True)

In [ ]:
icu_admission_results